Regularization - exercise 2

1. Load the diamond dataset (diamonds_preprocessed.csv) created in the "Linear Regression - exercise 2".
2. Import all necessary methods from libraries (or modules from these libraries).
3. Divide the dataset into training and test datasets.
5. Using the appropriate class, train a linear regression model in Lasso regularization.
    Verify the values of the coefficients - check which ones have been zeroed out by the model.
    Calculate the metrics R2, MAE, MSE and RMSE. Compare these values with the metrics the linear regression model legitimized.

Using steps 1-3:

    Using the appropriate class, train a linear regression model with Ridge regularization. Use different values of regularization strength.
    Calculate the R2, MAE, MSE and RMSE metrics. Compare these values with the metrics legitimized by the linear regression model without regularization and the model with Lasso regularization.


In [35]:
import pandas as pd

diamonds_df = pd.read_csv("data/diamonds_preprocessed.csv")
diamonds_df.head()

,carat,cut,color,clarity,depth,table,price,x,y,z
0,0.23,2,1,3,61.5,55.0,326,3.95,3.98,2.43
1,0.21,3,1,2,59.8,61.0,326,3.89,3.84,2.31
2,0.23,1,1,4,56.9,65.0,327,4.05,4.07,2.31
3,0.29,3,5,5,62.4,58.0,334,4.20,4.23,2.63
4,0.31,1,6,3,63.3,58.0,335,4.34,4.35,2.75


In [36]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
import numpy as np
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.compose import TransformedTargetRegressor
from sklearn.linear_model import Lasso




In [37]:
X = diamonds_df.drop('price', axis=1)
y = diamonds_df['price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [38]:
def lasso_regularization_metrics(X_train, X_test, y_train, y_test, alpha_values):
    results_lasso = []

    for alpha in alpha_values:
        lasso_pipeline = make_pipeline(
            StandardScaler(),
            Lasso(alpha=alpha, max_iter=50_000, tol=1e-4),
        )

        model = TransformedTargetRegressor(
            regressor=lasso_pipeline,
            transformer=StandardScaler(),
        )

        model.fit(X_train, y_train)
        y_train_pred_lasso = model.predict(X_train)
        y_test_pred_lasso = model.predict(X_test)
        lasso = model.regressor_.named_steps["lasso"]

        zeroed_features = [
            feature for feature, coefficient in zip(X_train.columns, lasso.coef_)
            if coefficient == 0
        ]

        mse_test = mean_squared_error(y_test, y_test_pred_lasso)
        results_lasso.append({
            "alpha": alpha,
            "r2_train": r2_score(y_train, y_train_pred_lasso),
            "r2_test": r2_score(y_test, y_test_pred_lasso),
            "mae_test": mean_absolute_error(y_test, y_test_pred_lasso),
            "mse_test": mse_test,
            "rmse_test": np.sqrt(mse_test),
            "zeroed_features": zeroed_features,
        })
    return pd.DataFrame(results_lasso)

alpha_values = [0.002, 0.02, 0.2, 2, 20, 200]
results = lasso_regularization_metrics(X_train, X_test, y_train, y_test, alpha_values)  
results  

,alpha,r2_train,r2_test,mae_test,mse_test,rmse_test,zeroed_features
0,0.002,0.884095,0.887265,856.577469,1.716880e+06,1310.297822,[y]
1,0.020,0.876383,0.878431,899.067083,1.851414e+06,1360.666611,"[x, y, z]"
2,0.200,0.808948,0.811617,1079.392753,2.868954e+06,1693.798636,"[cut, color, clarity, depth, table, x, y, z]"
3,2.000,0.000000,-0.000463,2982.982837,1.523644e+07,3903.388042,"[carat, cut, color, clarity, depth, table, x, ..."
4,20.000,0.000000,-0.000463,2982.982837,1.523644e+07,3903.388042,"[carat, cut, color, clarity, depth, table, x, ..."
5,200.000,0.000000,-0.000463,2982.982837,1.523644e+07,3903.388042,"[carat, cut, color, clarity, depth, table, x, ..."


## How the Lasso Metrics Function Works

The function fits a separate Lasso model for every value of `alpha`, then collects its scores and errors in a table. `alpha` controls the strength of the penalty: stronger regularization shrinks coefficients more, and Lasso can shrink some coefficients all the way to zero. The `zeroed_features` column lists those removed predictors.

### Why Use a Pipeline?

A pipeline chains preprocessing and modeling into one estimator. Here, `StandardScaler()` standardizes the predictor columns before Lasso fits. Lasso penalizes coefficient sizes, so features measured on very different scales could otherwise receive uneven penalties. The scaler learns its means and standard deviations from `X_train` when `model.fit(...)` runs, and reuses them for predictions. This keeps test data from influencing feature scaling.

### What Does `TransformedTargetRegressor` Do?

`TransformedTargetRegressor` wraps the pipeline and standardizes the target prices using the training targets. This puts large dollar amounts on a scale that can be easier for the coordinate-descent solver to optimize, helping with convergence. When `model.predict(...)` runs, the wrapper converts predictions back to the original price scale, so MAE, MSE, and RMSE are computed in meaningful units.

### Reading the Results

- `r2_train` shows how well each model fits the training examples; `r2_test` shows performance on held-out test examples.
- `mae_test`, `mse_test`, and `rmse_test` measure test-set error. MAE and RMSE are in price units; MSE is in squared price units.
- Compare scores together: increasing `alpha` may remove more features, but too much regularization can underfit and reduce R2.

For this exercise, the table compares each `alpha` on the test set. In a real project, choose `alpha` using cross-validation on the training data, then evaluate the chosen model on the test set once. Repeatedly choosing based on test scores makes the test set part of model selection.

In [40]:
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.linear_model import Ridge

# CV uses only the training split; the test split stays aside until final evaluation
cv = KFold(n_splits=5, shuffle=True, random_state=42)
cv_alpha_values = np.logspace(-4,2,10)

# Each model needs its own pipeline parameter name for alpha.
models_to_tune = [
    ("Lasso", Lasso(max_iter=100_000, tol=1e-4), "regressor__lasso__alpha"),
    ("Ridge", Ridge(max_iter=50_000, tol=1e-4), "regressor__ridge__alpha"),
]

cv_results = []
for model_name, estimator, alpha_parameter in models_to_tune:
    # Scale X inside the pipeline and scale y inside the target regressor.
    estimator_pipeline = make_pipeline(
        StandardScaler(),
        estimator
    )
    scaled_target_model = TransformedTargetRegressor(
        regressor=estimator_pipeline,
        transformer=StandardScaler(),
    )

    # GridSearchCV evaluates each alpha across the five training folds.
    search = GridSearchCV(
        estimator=scaled_target_model,
        param_grid={alpha_parameter: cv_alpha_values},
        scoring="neg_mean_absolute_error",
        cv=cv,
        refit=True,
    )
    search.fit(X_train, y_train)

    # refit=True trains the selected alpha on all training rows, now test once
    best_model = search.best_estimator_
    y_test_cv_pred = best_model.predict(X_test)
    mse_test_cv = mean_squared_error(y_test, y_test_cv_pred)

    cv_results.append(
        {
            "model": model_name,
            "best_alpha": search.best_params_[alpha_parameter],
            "mean_cv_mae": -search.best_score_,
            "test_r2": r2_score(y_test, y_test_cv_pred),
            "test_mae": mean_absolute_error(y_test, y_test_cv_pred),
            "test_mse": mse_test_cv,
            "test_rmse": np.sqrt(mse_test_cv),
        }
    )

cv_comparison = pd.DataFrame(cv_results)
cv_comparison


,model,best_alpha,mean_cv_mae,test_r2,test_mae,test_mse,test_rmse
0,Lasso,0.0001,864.280210,0.889233,846.547465,1.686913e+06,1298.812183
1,Ridge,1.0000,864.541786,0.889236,846.974857,1.686871e+06,1298.796053


## How Cross-Validation Selects Alpha

This cell chooses a regularization strength using only the training data, then checks the selected models on the held-out test data.

1. **Create folds:** `KFold` divides `X_train` and `y_train` into five parts. Each part takes a turn as the validation fold while the model fits on the other four.
2. **Try alpha values:** `GridSearchCV` fits each candidate `alpha` on these folds for both Lasso and Ridge. The parameter paths (`regressor__lasso__alpha` and `regressor__ridge__alpha`) tell the search which model parameter to vary inside the pipeline wrapped by `TransformedTargetRegressor`.
3. **Choose by validation MAE:** `scoring="neg_mean_absolute_error"` means the search scores negative MAE because scikit-learn selects the highest score. The code negates `best_score_` to display the usual positive MAE as `mean_cv_mae`.
4. **Refit the winner:** With `refit=True`, the chosen alpha's model is fitted again on all of `X_train` and `y_train`.
5. **Evaluate once on the test set:** The selected model predicts `X_test`; the reported test R2, MAE, MSE, and RMSE estimate performance on unseen data. The test results should not be used to choose alpha.

The feature scaler and target scaler are inside the estimator, so each cross-validation fold learns scaling values only from its own training portion. This prevents validation-fold information from leaking into that fold's model.